WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

Optimize a bilayer gel-elastomer cantilever rod to match a target centerline
profile read from a CSV of arc-length samples.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.interpolate as spy_interp

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.mma as _mma
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.default_plot_settings)

### Target profile

In [ ]:
target_df = pd.read_csv("crops/data/targets/sickle.csv")
arc_samples = target_df["s"].values
target_curve_fn = spy_interp.PchipInterpolator(
  arc_samples, target_df[["x", "y", "z"]].values
)

# The rod length is set by the target itself so the reference arc coordinate and the
# target parameter share a domain and no rescaling is needed anywhere downstream.
rod_length = float(arc_samples[-1])
print(f"rod length from target = {rod_length:.4f} m")

### Geometry

In [ ]:
# The cross-section is expressed through the slenderness rather than as an absolute
# size, so a single number controls how far the rod curls.
aspect_ratio = 8.0
breadth_to_height = 0.7

num_fibers = 12
num_elems = 150
num_nodes = num_elems + 1

rod_height = rod_length / aspect_ratio
rod_breadth = breadth_to_height * rod_height
layer_thickness = rod_height / num_fibers

# Layers stacked along d1 only. Every fiber therefore sits at y2 = 0.
d1_offsets = jnp.linspace(
  -0.5 * (rod_height - layer_thickness),
  0.5 * (rod_height - layer_thickness),
  num_fibers,
)
fiber_offsets = jnp.zeros((num_elems, num_fibers, 2)).at[:, :, 0].set(d1_offsets)
fiber_areas = jnp.full((num_elems, num_fibers), rod_breadth * layer_thickness)
fiber_heights = jnp.full((num_elems, num_fibers), layer_thickness)
fiber_thicknesses = jnp.full((num_elems, num_fibers), rod_breadth)

# Self MOI of each rectangular layer. Index 0 is about d1 and index 1 about d2.
fiber_centroid_moi = jnp.zeros((num_elems, num_fibers, 2))
fiber_centroid_moi = fiber_centroid_moi.at[:, :, 0].set(
  layer_thickness * rod_breadth**3 / 12.0
)
fiber_centroid_moi = fiber_centroid_moi.at[:, :, 1].set(
  rod_breadth * layer_thickness**3 / 12.0
)

fiber_mass_density = jnp.full((num_elems, num_fibers), 1100.0)
print(
  f"height {rod_height:.4f} m, breadth {rod_breadth:.4f} m, "
  f"layer {layer_thickness:.5f} m"
)

In [ ]:
rod = _mesher.build_rod_geometry(
  analytical_curve=lambda s: jnp.array([s, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, rod_length, num_nodes),
  fiber_offsets=fiber_offsets,
  fiber_areas=fiber_areas,
  fiber_mass_density=fiber_mass_density,
  fiber_centroid_moi=fiber_centroid_moi,
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)
target_coords = target_curve_fn(np.asarray(rod.arc_coords))

### Material and environment

In [ ]:
shear_modulus_gel, shear_modulus_elastomer = 1.0e6, 1.0e8
chi_gel, chi_elastomer = 0.3, 2.5

solvent = _fe.FlorySolvent(
  bath_temperature=298.0,
  molar_volume=1.8e-5,
  ref_chem_potential=0.0,
)
dry_chem_potential = -1.0e5
wet_chem_potential = -250.0

In [ ]:
num_steps = 20

ramp = jnp.linspace(0.0, 1.0, num_steps + 1)[1:] ** 0.03
chem_potentials = dry_chem_potential + (wet_chem_potential - dry_chem_potential) * ramp

### Boundary conditions

In [ ]:
clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
clamp = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
    node_targets=[np.array([0]) for _ in range(num_steps)],
    name=f"clamp_{dof}",
  )
  for dof in clamped_dofs
]

planar = [
  _bc.DirichletBC(
    values=(dof, [np.zeros(num_nodes) for _ in range(num_steps)]),
    node_targets=[np.arange(num_nodes) for _ in range(num_steps)],
    name=f"planar_{dof}",
  )
  for dof in (_utils.Translation.Z, _utils.Rotation.THETA_Y)
]
bc = _bc.process_boundary_conditions(rod, clamp + planar)

### Solvers and problem

In [ ]:
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 100,
    "threshold": 1e-6,
    "abs_threshold": 1e-6,
    "lam_min": 0.01,
    "line_search_max_iter": 12,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
  },
  linear_settings={"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-8},
)
problem = _fe.CosseratProblem(
  rod=rod,
  bc=bc,
  nr_solver=nr_solver,
  solvent=solvent,
  bisection_settings={
    "lower": 1e-6,
    "upper": 1.0 - 1e-6,
    "tol": 1e-6,
    "max_iter": 200,
  },
)

### Post-processing

`deformed_configuration` belongs in `mesher.py` rather than here. It depends only on
the rod and a displacement vector, not on the solver, so it works for any `u` from
any source and is testable without building a problem.

### Objective and constraint

In [ ]:
def solve_swelling(pseudodensity: jnp.ndarray) -> jnp.ndarray:
  """Converged displacement for a two-phase layout swollen to the bath potential.

  The pseudodensity linearly interpolates both the shear modulus and the Flory
  parameter between the gel and elastomer phases, so a value of one is fully gel.

  Args:
    pseudodensity: Array of shape `(num_elems, num_fibers)` in `[0, 1]`.

  Returns:
    Array of shape `(num_dofs,)` [m, rad].
  """
  shear_moduli = shear_modulus_gel * pseudodensity + shear_modulus_elastomer * (
    1.0 - pseudodensity
  )
  flory_parameters = chi_gel * pseudodensity + chi_elastomer * (1.0 - pseudodensity)

  u_star, _ = problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=shear_moduli,
    flory_parameters=flory_parameters,
    chem_potentials=chem_potentials,
  )
  return u_star


def compute_objective(design_vars: jnp.ndarray, thresh_beta: float):
  """Mean squared centerline error against the target, normalized by rod length.

  Both coordinates are compared, because a swelling rod elongates and a match in
  the transverse coordinate alone would be satisfied by the wrong arc length.

  Args:
    design_vars: Array of shape `(num_elems * num_fibers,)` in `[0, 1]`.
    thresh_beta: Sharpness of the projection pushing the design toward two phases.

  Returns:
    A tuple `(loss, (pseudodensity, u_star))`.
  """
  pseudodensity = _utils.threshold_filter(
    design_vars.reshape(num_elems, num_fibers), thresh_beta
  )
  u_star = solve_swelling(pseudodensity)
  nodal = u_star.reshape(num_nodes, rod.num_dofs_per_node)
  predicted = rod.node_coords[:, :2] + nodal[:, :2]
  error = predicted - jnp.asarray(target_coords[:, :2])
  loss = jnp.mean(jnp.einsum("nd, nd -> n", error, error)) / rod_length**2
  return loss, (pseudodensity, u_star)


objective_and_grad = jax.jit(
  jax.value_and_grad(compute_objective, argnums=0, has_aux=True)
)

In [ ]:
def grayness_constraint(
  pseudodensity: jnp.ndarray, grayness_allowed: float
) -> jnp.ndarray:
  """Penalize intermediate densities so the final design is nearly two-phase.

  Grayness is measured as `mean(x * (1 - x))`, which is zero for a design that is
  everywhere zero or one and maximal at one half. The constraint is written as
  `grayness - grayness_allowed <= 0`, so a positive value is a violation.

  Args:
    pseudodensity: Array of density values in `[0, 1]`.
    grayness_allowed: Slack on the permitted grayness.

  Returns:
    Scalar constraint value, non-positive when satisfied.
  """
  return jnp.mean(pseudodensity * (1.0 - pseudodensity)) - grayness_allowed


def compute_constraint(
  design_vars: jnp.ndarray, max_grayness: float, thresh_beta: float
):
  """Grayness constraint value and gradient in the shapes MMA expects."""

  def wrapper(x):
    projected = _utils.threshold_filter(x.reshape(num_elems, num_fibers), thresh_beta)
    return grayness_constraint(projected, max_grayness)

  value, gradient = jax.value_and_grad(wrapper)(design_vars)
  return np.array([[float(value)]]), np.asarray(gradient).reshape(1, -1)

### Optimization

In [ ]:
def run_optimization(
  init_density: jnp.ndarray,
  max_iter: int = 151,
  move_limit: float = 4e-3,
  plot_interval: int = 10,
):
  """Optimize the two-phase layout with MMA.

  Both the projection sharpness and the permitted grayness are annealed with the
  epoch, so the design is free to be gray early and is pushed toward two phases as
  it settles. The objective is rescaled once, after the first few epochs, so MMA's
  move limits and KKT tolerance act on a quantity of order unity.

  Args:
    init_density: Array of shape `(num_elems * num_fibers,)` in `[0, 1]`.
    max_iter: Maximum MMA epochs.
    move_limit: MMA move limit per epoch.
    plot_interval: Redraw the deformed shape every this many epochs. The shape
      comes from the auxiliary output of the objective, so it costs no extra solve.

  Returns:
    A tuple `(mma_state, history)`.
  """
  num_design_var = init_density.shape[0]
  mma_params = _mma.MMAParams(
    max_iter=max_iter,
    kkt_tol=1e-5,
    step_tol=1e-5,
    move_limit=move_limit,
    num_design_var=num_design_var,
    num_cons=1,
    lower_bound=np.zeros((num_design_var, 1)),
    upper_bound=np.ones((num_design_var, 1)),
  )
  mma_state = _mma.init_mma(np.asarray(init_density).reshape(-1, 1), mma_params)
  history = {"epoch": [], "objective": [], "constraint": []}
  obj_scale = 1.0

  while not mma_state.is_converged and mma_state.epoch < max_iter:
    max_grayness = max(0.02, 0.5 - 0.005 * mma_state.epoch)
    thresh_beta = min(64.0, 1.0 + 0.35 * mma_state.epoch)
    design_vars = jnp.asarray(mma_state.x).reshape(-1)

    (loss, (pseudodensity, u_star)), grad_obj = objective_and_grad(
      design_vars, thresh_beta
    )
    constraint, grad_constraint = compute_constraint(
      design_vars, max_grayness, thresh_beta
    )
    if mma_state.epoch == 10:
      obj_scale = 1.0 / float(jnp.abs(loss))

    history["epoch"].append(mma_state.epoch)
    history["objective"].append(float(loss))
    history["constraint"].append(float(constraint[0, 0]))
    print(f"epoch {mma_state.epoch:4d}  J {loss:.4E}  gray {constraint[0, 0]:+.4f}")

    if mma_state.epoch % plot_interval == 0:
      coords, directors, fiber_coords = (
        _mesher.extract_coords_and_director_from_deformation(rod, u_star)
      )
      ax = _viz.plot_rod_fiber_sections(
        fiber_coordinates=np.asarray(fiber_coords),
        directors=np.asarray(directors),
        fiber_heights=np.asarray(fiber_heights),
        fiber_thicknesses=np.asarray(fiber_thicknesses),
        fiber_field=np.asarray(pseudodensity),
        fiber_field_val_range=(0.0, 1.0),
        cmap=_viz.elast_gel_cmap,
        edge_width=0.0,
      )
      ax.plot(
        target_coords[:, 0],
        target_coords[:, 1],
        target_coords[:, 2],
        "k--",
        lw=1.2,
      )
      ax.view_init(elev=90, azim=-90)  # x-y plane
      # ax.view_init(elev=25, azim=-60)  # isometric
      ax.set_title(f"epoch {mma_state.epoch}   J = {loss:.3E}", fontsize=9)
      plt.show()
      plt.pause(1e-5)

    mma_state = _mma.update_mma(
      mma_state=mma_state,
      mma_params=mma_params,
      obj=np.array([float(loss) * obj_scale]),
      grad_obj=np.asarray(grad_obj, dtype=np.float64).reshape(-1, 1) * obj_scale,
      cons=constraint,
      grad_cons=grad_constraint,
    )
  return mma_state, history

In [ ]:
init_density = 0.5 * jnp.ones((num_elems * num_fibers,))
mma_state, history = run_optimization(init_density)

### Results

In [ ]:
fig, (ax_obj, ax_cons) = plt.subplots(1, 2, figsize=(11, 4))
ax_obj.semilogy(history["epoch"], history["objective"])
ax_obj.set_xlabel("MMA epoch")
ax_obj.set_ylabel("normalized shape error")
ax_cons.plot(history["epoch"], history["constraint"])
ax_cons.axhline(0.0, color="k", lw=0.5)
ax_cons.set_xlabel("MMA epoch")
ax_cons.set_ylabel("grayness constraint")
fig.tight_layout()
plt.show()

In [ ]:
final_density = _utils.threshold_filter(
  jnp.asarray(mma_state.x).reshape(num_elems, num_fibers), 64.0
)
u_final = solve_swelling(final_density)
coords, directors, fiber_coords = _mesher.extract_coords_and_director_from_deformation(
  rod, u_final
)

ax = _viz.plot_rod_fiber_sections(
  fiber_coordinates=np.asarray(fiber_coords),
  directors=np.asarray(directors),
  fiber_heights=np.asarray(fiber_heights),
  fiber_thicknesses=np.asarray(fiber_thicknesses),
  fiber_field=np.asarray(final_density),
  fiber_field_val_range=(0.0, 1.0),
  cmap=_viz.elast_gel_cmap,
  edge_width=0.0,
  plot_colorbar=True,
  colorbar_label="gel fraction",
)
ax.plot(
  target_coords[:, 0],
  target_coords[:, 1],
  target_coords[:, 2],
  "k--",
  lw=1.5,
  label="target",
)
ax.legend()
ax.view_init(elev=90, azim=-90)
plt.show()

nodal = np.asarray(u_final).reshape(num_nodes, rod.num_dofs_per_node)
print(
  f"max nodal rotation = "
  f"{np.degrees(np.abs(nodal[:, 3:]).max()):.1f} deg   (2 pi wall at 360)"
)